# 7. Sequence File Formats: FASTA, FASTQ, GenBank and GFF

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `fastq_summary.py`

Quality summary of the first 2,000 reads of SRR2584863, an Illumina run of E. coli from the long term evolution experiment (NCBI SRA).


In [ ]:
# Quality summary of the first 2,000 reads of SRR2584863, an Illumina run
# of E. coli from the long term evolution experiment (NCBI SRA).
import gzip

from Bio import SeqIO

reads = bases = high_quality = poor_reads = 0

with gzip.open("reads.fastq.gz", "rt") as handle:
    for record in SeqIO.parse(handle, "fastq"):
        qualities = record.letter_annotations["phred_quality"]
        reads += 1
        bases += len(qualities)
        high_quality += sum(q >= 30 for q in qualities)
        if sum(qualities) / len(qualities) < 20:
            poor_reads += 1

print(f"reads                    {reads:8,}")
print(f"bases                    {bases:8,}")
print(f"bases with quality >= 30 {high_quality / bases:8.1%}")
print(f"reads averaging below 20 {poor_reads:8,}")

## `gff_genes.py`

The genes of E. coli K-12 MG1655 from NCBI's GFF3 annotation.


In [ ]:
# The genes of E. coli K-12 MG1655 from NCBI's GFF3 annotation.
import pandas as pd

COLUMNS = [
    "seqid",
    "source",
    "type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes",
]


def parse_attributes(text: str) -> dict[str, str]:
    """ID=gene-b0344;Name=lacZ;... as a dictionary."""
    return dict(pair.split("=", 1) for pair in text.split(";"))


gff = pd.read_csv(
    "ecoli_k12.gff3.gz", sep="\t", comment="#", names=COLUMNS
)
genes = gff[gff["type"] == "gene"].copy()
attributes = genes["attributes"].map(parse_attributes)
genes["name"] = attributes.map(lambda a: a["Name"])
genes["biotype"] = attributes.map(lambda a: a["gene_biotype"])
genes["length"] = genes["end"] - genes["start"] + 1

print(genes["biotype"].value_counts().to_string())
print()
lac = genes[genes["name"].str.startswith("lac")]
print(lac[["name", "start", "end", "strand", "length"]].to_string())

## `lac_proteins.py`

Write the proteins of the lac genes, from their GenBank annotation, as a FASTA file.


In [ ]:
# Write the proteins of the lac genes, from their GenBank annotation, as
# a FASTA file.
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

region = SeqIO.read("../ch06/lac_region.gb", "genbank")

proteins = []
for feature in region.features:
    if feature.type != "CDS":
        continue
    gene = feature.qualifiers["gene"][0]
    if not gene.startswith("lac"):
        continue
    protein = SeqRecord(
        Seq(feature.qualifiers["translation"][0]),
        id=feature.qualifiers["protein_id"][0],
        description=f"{gene} {feature.qualifiers['product'][0]}",
    )
    proteins.append(protein)

count = SeqIO.write(proteins, "lac_proteins.fasta", "fasta")
print(f"wrote {count} proteins to lac_proteins.fasta")